In [1]:
import pandas as pd
import numpy as np
import math

print("1. Loading Detailed Box Scores...")
w_details = pd.read_csv('WRegularSeasonDetailedResults.csv')
w_details = w_details[w_details['Season'] >= 2010].copy()
w_details.sort_values(['Season', 'DayNum'], inplace=True)

# ==========================================
# PHASE 1: SIMULATE ELO & DAILY RANKINGS
# ==========================================
print("2. Simulating 15 years of Elo Ratings & Ranks...")

elo_dict = {}
records = []
current_season = 0

for season, day_data in w_details.groupby('Season'):
    # Offseason Reversion: Pull teams 25% back to 1500 average between seasons
    for team in elo_dict:
        elo_dict[team] = (elo_dict[team] * 0.75) + (1500 * 0.25)

    for day, games in day_data.groupby('DayNum'):
        # Sort current Elos to calculate everyone's Rank on this specific day
        sorted_elos = sorted(elo_dict.values(), reverse=True)
        day_updates = {}

        for idx, row in games.iterrows():
            w_team = row['WTeamID']
            l_team = row['LTeamID']

            if w_team not in elo_dict: elo_dict[w_team] = 1500
            if l_team not in elo_dict: elo_dict[l_team] = 1500

            w_elo = elo_dict[w_team]
            l_elo = elo_dict[l_team]

            # Calculate Opponent's Rank for Quadrant logic later
            w_rank = sorted_elos.index(w_elo) + 1 if w_elo in sorted_elos else len(sorted_elos) + 1
            l_rank = sorted_elos.index(l_elo) + 1 if l_elo in sorted_elos else len(sorted_elos) + 1

            # Elo Math
            expected_w = 1 / (1 + 10 ** ((l_elo - w_elo) / 400))
            expected_l = 1 / (1 + 10 ** ((w_elo - l_elo) / 400))

            mov = row['WScore'] - row['LScore']
            mov_mult = math.log(mov + 1) * (2.2 / (((w_elo - l_elo) * 0.001) + 2.2))
            mov_mult = max(1.0, mov_mult)

            # Update Elos (K=20)
            day_updates[w_team] = w_elo + (20 * mov_mult * (1 - expected_w))
            day_updates[l_team] = l_elo + (20 * mov_mult * (0 - expected_l))

            records.append({
                'GameID': idx,
                'W_Entering_Elo': w_elo,
                'L_Entering_Elo': l_elo,
                'W_Opp_Rank': l_rank, # The Winner played against the Loser's Rank
                'L_Opp_Rank': w_rank
            })

        # Apply the new Elos at the end of the day
        for team, new_elo in day_updates.items():
            elo_dict[team] = new_elo

elo_df = pd.DataFrame(records).set_index('GameID')
w_details = w_details.join(elo_df)

# ==========================================
# PHASE 2: CALCULATE ADVANCED STATS
# ==========================================
print("3. Calculating Tempo-Free Stats...")
w_details['WPos'] = w_details['WFGA'] - w_details['WOR'] + w_details['WTO'] + (0.475 * w_details['WFTA'])
w_details['LPos'] = w_details['LFGA'] - w_details['LOR'] + w_details['LTO'] + (0.475 * w_details['LFTA'])

w_details['W_ORtg'] = (w_details['WScore'] / w_details['WPos']) * 100
w_details['W_DRtg'] = (w_details['LScore'] / w_details['LPos']) * 100
w_details['L_ORtg'] = (w_details['LScore'] / w_details['LPos']) * 100
w_details['L_DRtg'] = (w_details['WScore'] / w_details['WPos']) * 100

# ==========================================
# PHASE 3: MELT INTO TEAM-GAME FORMAT
# ==========================================
print("4. Pivoting to Team-Game Histories & Classifying Quadrants...")

# Helpers to inverse locations for the Loser
loc_map = {'H': 'A', 'A': 'H', 'N': 'N'}

# Winners Frame
winners = w_details[['Season', 'DayNum', 'WTeamID', 'WLoc', 'WScore', 'LScore', 'W_ORtg', 'W_DRtg', 'W_Entering_Elo', 'W_Opp_Rank']].copy()
winners.columns = ['Season', 'DayNum', 'TeamID', 'Loc', 'PF', 'PA', 'Game_ORtg', 'Game_DRtg', 'Entering_Elo', 'Opp_Rank']
winners['Win'] = 1

# Losers Frame
losers = w_details[['Season', 'DayNum', 'LTeamID', 'WLoc', 'LScore', 'WScore', 'L_ORtg', 'L_DRtg', 'L_Entering_Elo', 'L_Opp_Rank']].copy()
losers['WLoc'] = losers['WLoc'].map(loc_map) # If Winner was Home, Loser was Away
losers.columns = ['Season', 'DayNum', 'TeamID', 'Loc', 'PF', 'PA', 'Game_ORtg', 'Game_DRtg', 'Entering_Elo', 'Opp_Rank']
losers['Win'] = 0

w_history = pd.concat([winners, losers]).sort_values(['Season', 'TeamID', 'DayNum'])
w_history['PointDiff'] = w_history['PF'] - w_history['PA']

# ==========================================
# PHASE 4: ASSIGN NCAA QUADRANTS
# ==========================================
def get_quadrant(row):
    rank = row['Opp_Rank']
    loc = row['Loc']
    if loc == 'H':
        if rank <= 30: return 'Q1'
    elif loc == 'N':
        if rank <= 50: return 'Q1'
    elif loc == 'A':
        if rank <= 75: return 'Q1'
    return 'Q234' # Everything else is standard games

w_history['Quad'] = w_history.apply(get_quadrant, axis=1)

# Create binary Win/Loss columns for the Quadrants
w_history['Q1_W'] = ((w_history['Quad'] == 'Q1') & (w_history['Win'] == 1)).astype(int)
w_history['Q1_L'] = ((w_history['Quad'] == 'Q1') & (w_history['Win'] == 0)).astype(int)
w_history['Q234_W'] = ((w_history['Quad'] == 'Q234') & (w_history['Win'] == 1)).astype(int)
w_history['Q234_L'] = ((w_history['Quad'] == 'Q234') & (w_history['Win'] == 0)).astype(int)

# ==========================================
# PHASE 5: CUMULATIVE ROLLING AVERAGES
# ==========================================
print("5. Calculating Rolling Performance (Preventing Data Leakage)...")

gb = w_history.groupby(['Season', 'TeamID'])

# Cumulative Quadrant Records
w_history['Avg_ORtg'] = gb['Game_ORtg'].expanding().mean().reset_index(0, drop=True).reset_index(0, drop=True)
w_history['Avg_DRtg'] = gb['Game_DRtg'].expanding().mean().reset_index(0, drop=True).reset_index(0, drop=True)
w_history['Avg_PointDiff'] = gb['PointDiff'].expanding().mean().reset_index(0, drop=True).reset_index(0, drop=True)
w_history['Std_PointDiff'] = gb['PointDiff'].expanding().std().reset_index(0, drop=True).reset_index(0, drop=True)

w_history['Cum_Q1_W'] = gb['Q1_W'].cumsum()
w_history['Cum_Q1_L'] = gb['Q1_L'].cumsum()
w_history['Cum_Q234_W'] = gb['Q234_W'].cumsum()
w_history['Cum_Q234_L'] = gb['Q234_L'].cumsum()

# SHIFT EVERYTHING BY 1! (So we only know stats prior to today's game)
shift_cols = ['Avg_ORtg', 'Avg_DRtg', 'Avg_PointDiff', 'Std_PointDiff', 'Cum_Q1_W', 'Cum_Q1_L', 'Cum_Q234_W', 'Cum_Q234_L']
for col in shift_cols:
    w_history[f'Entering_{col}'] = gb[col].shift(1)

# Standard Dev is NaN for Game 1 and Game 2. Fill with 0.
w_history['Entering_Std_PointDiff'] = w_history['Entering_Std_PointDiff'].fillna(0)

# Isolate only the Entering metrics
final_cols = ['Season', 'DayNum', 'TeamID', 'Entering_Elo'] + [f'Entering_{col}' for col in shift_cols]
w_master_features = w_history[final_cols].copy()

# Drop Game 1s where teams have no entering history
w_master_features.dropna(inplace=True)

w_master_features.to_csv('WOMENS_MASTER_FEATURES.csv', index=False)

print("\nSUCCESS! Women's Master Feature Data Saved!")
print(w_master_features.head())

1. Loading Detailed Box Scores...
2. Simulating 15 years of Elo Ratings & Ranks...
3. Calculating Tempo-Free Stats...
4. Pivoting to Team-Game Histories & Classifying Quadrants...
5. Calculating Rolling Performance (Preventing Data Leakage)...

SUCCESS! Women's Master Feature Data Saved!
      Season  DayNum  TeamID  Entering_Elo  Entering_Avg_ORtg  \
130     2010      12    3102   1470.042677          66.187050   
382     2010      18    3102   1436.472482          72.121801   
608     2010      23    3102   1416.303571          82.296229   
722     2010      25    3102   1393.444753          78.455239   
1025    2010      32    3102   1367.706498          78.193314   

      Entering_Avg_DRtg  Entering_Avg_PointDiff  Entering_Std_PointDiff  \
130           93.457944              -19.000000                0.000000   
382          109.012709              -25.500000                9.192388   
608          111.780536              -19.666667               12.013881   
722          108.126

In [3]:
import pandas as pd

print("1. Structuring Women's Matchups with symmetric perspectives...")
w_games = pd.read_csv('WRegularSeasonDetailedResults.csv')
w_games = w_games[w_games['Season'] >= 2010].copy()

# Perspective 1: Team A is Winner
w_win_a = w_games.copy()
w_win_a['TeamA_ID'] = w_win_a['WTeamID']
w_win_a['TeamB_ID'] = w_win_a['LTeamID']
w_win_a['TeamA_Score'] = w_win_a['WScore']
w_win_a['TeamB_Score'] = w_win_a['LScore']
w_win_a['Target'] = 1

# Perspective 2: Team A is Loser
w_lose_a = w_games.copy()
w_lose_a['TeamA_ID'] = w_lose_a['LTeamID']
w_lose_a['TeamB_ID'] = w_lose_a['WTeamID']
w_lose_a['TeamA_Score'] = w_lose_a['LScore']
w_lose_a['TeamB_Score'] = w_lose_a['WScore']
w_lose_a['Target'] = 0

# Combine them into the unified ML dataset
w_ml_data = pd.concat([w_win_a, w_lose_a], ignore_index=True)
w_ml_data['ScoreDiff'] = w_ml_data['TeamA_Score'] - w_ml_data['TeamB_Score']
w_ml_data = w_ml_data[['Season', 'DayNum', 'TeamA_ID', 'TeamB_ID', 'ScoreDiff', 'Target']].copy()

print("2. Loading Women's Master Features...")
w_master = pd.read_csv('WOMENS_MASTER_FEATURES.csv')

print("3. Merging features onto Team A and Team B...")
# CRITICAL FIX: Sort strictly by DayNum before using merge_asof
w_ml_data.sort_values('DayNum', inplace=True)
w_master.sort_values('DayNum', inplace=True)

def attach_w_features(df, team_col, prefix):
    df = pd.merge_asof(df, w_master, on='DayNum', left_by=['Season', team_col], right_by=['Season', 'TeamID'], direction='backward')
    df.drop(columns=['TeamID'], inplace=True)
    rename_dict = {col: f"{prefix}_{col}" for col in df.columns if col not in ['Season', 'DayNum', 'TeamA_ID', 'TeamB_ID', 'ScoreDiff', 'Target'] and not col.startswith('TeamA_') and not col.startswith('TeamB_')}
    df.rename(columns=rename_dict, inplace=True)
    return df

w_ml_data = attach_w_features(w_ml_data, 'TeamA_ID', 'TeamA')
w_ml_data = attach_w_features(w_ml_data, 'TeamB_ID', 'TeamB')

print("4. Calculating feature differences...")
# Re-sort for human readability
w_ml_data.sort_values(['Season', 'DayNum'], inplace=True)

features_to_diff_w = [
    'Entering_Elo', 'Entering_Avg_ORtg', 'Entering_Avg_DRtg',
    'Entering_Cum_Q1_W', 'Entering_Cum_Q1_L', 'Entering_Cum_Q234_W', 'Entering_Cum_Q234_L',
    'Entering_Avg_PointDiff'
]

for stat in features_to_diff_w:
    if f'TeamA_{stat}' in w_ml_data.columns and f'TeamB_{stat}' in w_ml_data.columns:
        w_ml_data[f'Diff_{stat}'] = w_ml_data[f'TeamA_{stat}'] - w_ml_data[f'TeamB_{stat}']

# Drop the raw columns to keep the dataset lean and focused on the differences
cols_to_drop = [f'TeamA_{stat}' for stat in features_to_diff_w] + [f'TeamB_{stat}' for stat in features_to_diff_w]
w_ml_data.drop(columns=cols_to_drop, inplace=True, errors='ignore')
w_ml_data.dropna(inplace=True)

output_filename = 'W_ULTIMATE_XGBOOST_TRAINING_DATA.csv'
w_ml_data.to_csv(output_filename, index=False)
print(f"\nSUCCESS! Master dataset saved as '{output_filename}'")

1. Structuring Women's Matchups with symmetric perspectives...
2. Loading Women's Master Features...
3. Merging features onto Team A and Team B...
4. Calculating feature differences...

SUCCESS! Master dataset saved as 'W_ULTIMATE_XGBOOST_TRAINING_DATA.csv'


In [5]:
import pandas as pd
import numpy as np
import xgboost as xgb
import matplotlib.pyplot as plt
import joblib
from scipy.stats import norm
from sklearn.metrics import mean_squared_error

print("1. Loading the Women's Ultimate Regression Dataset...")
w_df = pd.read_csv('W_ULTIMATE_XGBOOST_TRAINING_DATA.csv')

w_train = w_df[w_df['Season'] <= 2025].copy()
w_valid = w_df[w_df['Season'] == 2026].copy()

drop_cols = ['Season', 'DayNum', 'TeamA_ID', 'TeamB_ID', 'Target', 'ScoreDiff']
X_train_w = w_train.drop(columns=drop_cols)
y_train_spread_w = w_train['ScoreDiff']

X_valid_w = w_valid.drop(columns=drop_cols)
y_valid_spread_w = w_valid['ScoreDiff']
y_valid_binary_w = w_valid['Target']

print("\n2. Training Model and Validating on 2026 Regular Season...")
w_model = xgb.XGBRegressor(
    n_estimators=2000, max_depth=4, learning_rate=0.02,
    subsample=0.5, colsample_bytree=0.5, eval_metric='mae',
    early_stopping_rounds=50, random_state=42
)

w_model.fit(
    X_train_w, y_train_spread_w,
    eval_set=[(X_train_w, y_train_spread_w), (X_valid_w, y_valid_spread_w)],
    verbose=False
)

best_iteration_w = w_model.best_iteration
print(f"   -> Women's Model perfectly tuned to the 2026 Meta at Tree {best_iteration_w}!")

# ----- NEW: SAVE TO DISK -----
w_model.save_model('womens_xgboost_model.json')
joblib.dump({'best_iteration': best_iteration_w}, 'w_model_meta.pkl')
print("✅ Women's Model and metadata successfully saved to disk!")

1. Loading the Women's Ultimate Regression Dataset...

2. Training Model and Validating on 2026 Regular Season...
   -> Women's Model perfectly tuned to the 2026 Meta at Tree 857!
✅ Women's Model and metadata successfully saved to disk!


In [6]:
import pandas as pd
import numpy as np
import xgboost as xgb
import joblib
from scipy.stats import norm

# ----- NEW: LOAD FROM DISK -----
w_model = xgb.XGBRegressor()
w_model.load_model('womens_xgboost_model.json')
best_iteration_w = joblib.load('w_model_meta.pkl')['best_iteration']

print("Loading Kaggle's Stage 2 Submission Template...")
sub = pd.read_csv('SampleSubmissionStage2.csv')
sub[['Season', 'TeamA_ID', 'TeamB_ID']] = sub['ID'].str.split('_', expand=True).astype(int)

women_sub = sub[sub['TeamA_ID'] >= 3000].copy()
print(f"Structuring {len(women_sub)} Women's Matchups...")

w_features = pd.read_csv('WOMENS_MASTER_FEATURES.csv')
w_final_2026 = w_features[w_features['Season'] == 2026].groupby('TeamID').tail(1).copy()
w_final_2026.drop(columns=['DayNum', 'Season'], inplace=True, errors='ignore')

women_sub = pd.merge(women_sub, w_final_2026, left_on='TeamA_ID', right_on='TeamID', how='left')
women_sub.rename(columns={col: f'TeamA_{col}' for col in w_final_2026.columns if col != 'TeamID'}, inplace=True)
women_sub.drop(columns=['TeamID'], inplace=True)

women_sub = pd.merge(women_sub, w_final_2026, left_on='TeamB_ID', right_on='TeamID', how='left')
women_sub.rename(columns={col: f'TeamB_{col}' for col in w_final_2026.columns if col != 'TeamID'}, inplace=True)
women_sub.drop(columns=['TeamID'], inplace=True)

features_to_diff_w = [
    'Entering_Elo', 'Entering_Avg_ORtg', 'Entering_Avg_DRtg',
    'Entering_Cum_Q1_W', 'Entering_Cum_Q1_L', 'Entering_Cum_Q234_W', 'Entering_Cum_Q234_L',
    'Entering_Avg_PointDiff'
]

for stat in features_to_diff_w:
    if f'TeamA_{stat}' in women_sub.columns and f'TeamB_{stat}' in women_sub.columns:
        women_sub[f'Diff_{stat}'] = women_sub[f'TeamA_{stat}'] - women_sub[f'TeamB_{stat}']

cols_to_drop_w = [f'TeamA_{stat}' for stat in features_to_diff_w] + [f'TeamB_{stat}' for stat in features_to_diff_w]
women_sub.drop(columns=cols_to_drop_w, inplace=True, errors='ignore')
women_sub.fillna(0, inplace=True)

print("Predicting Women's Bracket...")
X_women = women_sub.drop(columns=['ID', 'Pred', 'Season', 'TeamA_ID', 'TeamB_ID'])
women_spreads = w_model.predict(X_women, iteration_range=(0, best_iteration_w + 1))

women_sub['Pred'] = norm.cdf(women_spreads, loc=0, scale=12.3)
women_sub[['ID', 'Pred']].to_csv('womens_stage2_predictions.csv', index=False)
print("🏆 Women's Stage 2 predictions saved!")

Loading Kaggle's Stage 2 Submission Template...
Structuring 65703 Women's Matchups...
Predicting Women's Bracket...
🏆 Women's Stage 2 predictions saved!
